Yiscah Mark - FI

Project Objectives:
The point of this project was to create a program that ingests various resumes and correctly categorizes them into the appropriate categories. I used a very large dataset from Kaggle with many prelabeled resumes, so the program could train itself to recognize and sort various resumes.

Programmatic Achievement
When researching the dataset, it appears very simple and easy to work with. As soon as I started, I realized how unstructured and hard to work with it was, among other issues. Not one to give up, I pushed through without changing data in the middle. 
After achieving a very low accuracy of 53%, I improved the program by rewriting the ingestion logic to perform a safe split of the raw CSV lines and correctly organize them into a structured Pandas DataFrame.
I converted the text into numerical data so the computer could process it using distance calculations. A computer can’t really understand language unless spoonfed in technical bite-sized pieces. When words are converted to vectors and numerical arrays, the computer can perform complex calculations to determine the distance and similarity between texts.

Observations:
The famous saying about data is “garbage in, garbage out”. People expect data to be a magical answer to everything, a prediction of the future… While there is a lot of potential for data, if it isn’t of good quality, not much can be extracted from it. The beginning of my project was a great example of what not to do. My model scored 53% accuracy. This was likely due to label misalignment. I fixed this issue by implementing a safesplit function that specifically handles encoding errors. This step was essential to correctly align the resume text with its prospective category. 
There is a mathematical way to actually measure similarity between texts. The Euclidean distance measures the distance between vectors to determine similarity. A similarity of 0 indicates perfect similarity; thus, a value of 1.3 indicates that the resume classified as taught is very different from the one for Apparel.
A critical piece of my NLP pipeline was to remove background noise and normalize the text. I removed all stopwords because they don’t give a unique meaning to the text. I also removed special characters because they made the text very messy, and I lowercased all the words to unify them. I also ran the text through the lemmatizer, which reduces all words to their root. By unifying all the text, the model can more easily detect patterns and similarities.
This program successfully sorts resumes into categories. I learned how important data structure is and how to clean and remove noise from text so that similarities and nuances are clearer. This achieves the object of turning unstructured data into workable material to provide a scalable solution for processing large volumes of professional documents.




This notebook is a resume classifier. the model is trained to detect, based on document similarity resumes of specific categories. I used a very large data set from kaggle with many resumes that are already labled by category to train and test the model.

This cell uploads the dataset. after trying to use the dataset i realized that it was very messy and unorganized my model only had a .53 accuracy, so i went back and uploaded it like this.

In [2]:

import pandas as pd

# Load file as plain text
with open("resumeclass.csv", "r", encoding="utf-8", errors="ignore") as f:
    lines = f.readlines()




this cell organizes the information on the resume into correct columns

In [3]:
# Put into DataFrame
df = pd.DataFrame({"raw": [line.strip() for line in lines]})

# Safe split into 3 parts
def safe_split(line):
    parts = line.split(",", 2)
    if len(parts) < 3:
        return None, None, None
    return parts[0], parts[1], parts[2]

df["resume_id"], df["category"], df["resume_text"] = zip(*df["raw"].apply(safe_split))
df = df.dropna(subset=["resume_text"])
df = df[["resume_id", "category", "resume_text"]]



In [4]:
# Clean category labels
df["category"] = df["category"].str.strip().str.upper()


Before applying any models to the data we need to normalize the text. normalizing the text is basically cleaning it, and removing 'backround noise' so the model can extract the text similarities. this normalizer lowercases all capital letters so they are unified, removes punctuation, removes stopwords, (very common words that don;t make a difference to the meaning of the text, for classifying's sake). This model also lemmatizes.

In [5]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')


stop_words = set(stopwords.words("english"))

resume_stopwords = [
    "resume", "objective", "summary", "professional", "skills",
    "experience", "responsibilities", "work", "team", "company",
    "role", "project", "projects", "task", "tasks"
]



stop_words.update(resume_stopwords)
lemmatizer = WordNetLemmatizer()

def normalize_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    words = text.split()
    words = [w for w in words if w not in stop_words]
    words = [lemmatizer.lemmatize(w) for w in words]
    return " ".join(words)





[nltk_data] Downloading package stopwords to
[nltk_data]     /home/f8bded8e-df7e-4491-93d9-
[nltk_data]     90b7721edcf0/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /home/f8bded8e-df7e-4491-93d9-
[nltk_data]     90b7721edcf0/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     /home/f8bded8e-df7e-4491-93d9-
[nltk_data]     90b7721edcf0/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


just creating a dataframe with the old and new normalized text side by side

In [6]:
df["cleanresumetext"] = df["resume_text"].apply(normalize_text)

In [7]:
df.head()

,resume_id,category,resume_text,cleanresumetext
0,"﻿""resume_id",CATEGORY,"resume_text,skills_list,experience_years""",text list year
1,"""1",HR,hr administrator marketing associate hr admini...,hr administrator marketing associate hr admini...
2,"""2",HR,hr specialist us hr operations summary versati...,hr specialist u hr operation versatile medium ...
3,"""3",HR,hr director summary over years experience in r...,hr director year recruiting plus year human re...
4,"""4",HR,hr specialist summary dedicated driven and dyn...,hr specialist dedicated driven dynamic year cu...


#had to remove a category that there was only one of.

In [8]:
df = df[df["category"] != "CATEGORY"]

In [9]:
df["category"].value_counts()

category
ADVOCATE                              128
SALES                                 121
HR                                    120
INFORMATION-TECHNOLOGY                120
BUSINESS-DEVELOPMENT                  120
ACCOUNTANT                            118
ENGINEERING                           118
CHEF                                  118
FINANCE                               117
FITNESS                               117
AVIATION                              116
CONSULTANT                            115
BANKING                               115
HEALTHCARE                            115
CONSTRUCTION                          112
PUBLIC-RELATIONS                      111
ARTS                                  109
DESIGNER                              107
TEACHER                               102
APPAREL                                97
DIGITAL-MEDIA                          96
AGRICULTURE                            63
AUTOMOBILE                             36
FULL STACK DEVELOPER     

A very important part is to choose the right amount of groups, it seems to me like there are too many groups that are suggesting similar jobs. this may possibly mess up the data set later on. for example "data science" and "data scientist"

split the data into 80% to train the model and 20% to test it.

In [10]:


from sklearn.model_selection import train_test_split

X = df["cleanresumetext"]
y = df["category"]

x_train, x_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


i performed the bag of words model on my data, it is a table with every word that appears in all the documents and the amount of times this word appears in each document. It is used to extract document similarity.

In [11]:
from sklearn.feature_extraction.text import CountVectorizer

# Bag‑of‑Words vectorizer
bow_vectorizer = CountVectorizer(min_df=3)

# Fit on training data, transform both sets
bow_train = bow_vectorizer.fit_transform(x_train)
bow_test = bow_vectorizer.transform(x_test)

bow_train.shape, bow_test.shape


((2268, 11716), (568, 11716))

this is the Naive Bayes model, the accuracy is very low in this data.

In [12]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report

# Train Naive Bayes
bow_model = MultinomialNB()
bow_model.fit(bow_train, y_train)

# Predict
bow_pred = bow_model.predict(bow_test)

print("=== BAG OF WORDS + NAIVE BAYES ===")
print("Accuracy:", accuracy_score(y_test, bow_pred))
print(classification_report(y_test, bow_pred))

=== BAG OF WORDS + NAIVE BAYES ===
Accuracy: 0.5316901408450704
                                    precision    recall  f1-score   support

                        ACCOUNTANT       0.68      0.71      0.69        24
                          ADVOCATE       0.30      0.35      0.32        26
                       AGRICULTURE       0.62      0.38      0.48        13
                           APPAREL       0.47      0.37      0.41        19
                              ARTS       0.33      0.32      0.33        22
                AUTOMATION TESTING       0.50      1.00      0.67         1
                        AUTOMOBILE       0.00      0.00      0.00         7
                          AVIATION       0.54      0.61      0.57        23
                 BACKEND DEVELOPER       1.00      0.50      0.67         4
                           BANKING       0.63      0.52      0.57        23
                        BLOCKCHAIN       1.00      1.00      1.00         1
                       

/opt/conda/envs/anaconda-ai-2025.12-py312/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/conda/envs/anaconda-ai-2025.12-py312/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/conda/envs/anaconda-ai-2025.12-py312/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, 

##this model performed not perfectly but better than the last one. The Tf-Idf model is more accurate because it takes into account the weight of each word and how much meaning that word lends to the classification.

In [13]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC

tfidf_vectorizer = TfidfVectorizer(ngram_range=(1,3), min_df=3, max_df = 0.9, max_features=40000)
tfidf_train = tfidf_vectorizer.fit_transform(x_train)
tfidf_test = tfidf_vectorizer.transform(x_test)

svm_model = LinearSVC()
svm_model.fit(tfidf_train, y_train)

svm_pred = svm_model.predict(tfidf_test)

print("=== TF-IDF + SVM ===")
print("Accuracy:", accuracy_score(y_test, svm_pred))
print(classification_report(y_test, svm_pred))


=== TF-IDF + SVM ===
Accuracy: 0.727112676056338
                                    precision    recall  f1-score   support

                        ACCOUNTANT       0.76      0.79      0.78        24
                          ADVOCATE       0.63      0.73      0.68        26
                       AGRICULTURE       0.67      0.31      0.42        13
                           APPAREL       0.69      0.58      0.63        19
                              ARTS       0.64      0.41      0.50        22
                AUTOMATION TESTING       0.33      1.00      0.50         1
                        AUTOMOBILE       0.50      0.14      0.22         7
                          AVIATION       0.74      0.74      0.74        23
                 BACKEND DEVELOPER       1.00      1.00      1.00         4
                           BANKING       0.82      0.61      0.70        23
                        BLOCKCHAIN       1.00      1.00      1.00         1
                               BPO    

/opt/conda/envs/anaconda-ai-2025.12-py312/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/conda/envs/anaconda-ai-2025.12-py312/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/conda/envs/anaconda-ai-2025.12-py312/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, 

#i wasn't so happy with the accuracy of my model so i went and added a list of resume specific stopwords but it didn't change the accuracy

#i then went to change the max_df, max_features to try to improve the accuracy i adjusted the maxfeatures untill i found the best one bur it only improved the model by less then 1%

Milestone 4

The following is a model that groups the datasets into categories, highlighting important words that appear often in each group. The grouping is done while ignoring the original labels. This is important to see how different the data categories are for the dataset.

I only used a sample of 500 resumes from the data set for technical reasons.

In [20]:
# 1. DATA DECISION: Sample the data to ensure memory efficiency in the web environment 
# This satisfies the milestone's requirement to "make decisions about the data itself".
df_sample = df.sample(n=500, random_state=42).reset_index(drop=True)

In [21]:
# 2. VECTORIZATION: Create a TF-IDF matrix specifically for this sample
from sklearn.feature_extraction.text import TfidfVectorizer

# Vectorizing the 'cleanresumetext' column you created earlier
tfidf_vectorizer = TfidfVectorizer(max_features=5000)
tfidf_matrix = tfidf_vectorizer.fit_transform(df_sample['cleanresumetext'])


In [22]:
# 3. CLUSTERING: Use K-Means to group the text and uncover "common term relationships"
from sklearn.cluster import KMeans

num_clusters = 5 # You can change this number to see different groupings
km = KMeans(n_clusters=num_clusters, random_state=42, n_init=10)
km.fit(tfidf_matrix)

,n_clusters,5
,init,'k-means++'
,n_init,10
,max_iter,300
,tol,0.0001
,verbose,0
,random_state,42
,copy_x,True
,algorithm,'lloyd'


In [23]:
# 4. VIEW TERM RELATIONSHIPS: Print the top 10 words that define each cluster
print("=== Top Terms per Cluster ===")
order_centroids = km.cluster_centers_.argsort()[:, ::-1]
terms = tfidf_vectorizer.get_feature_names_out()

for i in range(num_clusters):
    print(f"Cluster {i}:")
    top_words = [terms[ind] for ind in order_centroids[i, :10]]
    print(", ".join(top_words))
    print("-" * 50)


=== Top Terms per Cluster ===
Cluster 0:
accounting, financial, account, reconciliation, accountant, tax, finance, ledger, statement, monthly
--------------------------------------------------
Cluster 1:
exprience, cloud, developer, python, data, implementing, detail, using, contributed, learning
--------------------------------------------------
Cluster 2:
student, state, medium, marketing, city, design, name, digital, teacher, art
--------------------------------------------------
Cluster 3:
system, state, city, server, engineering, management, food, equipment, name, construction
--------------------------------------------------
Cluster 4:
customer, sale, state, service, city, management, patient, business, name, client
--------------------------------------------------


The model identified 5 clusters, the top cluster looks pretty clearly like accounting, but the bottom three all have city, state, and name which makes it hard to differenciate. It might just be general words that don't make a difference. It is possible that this is why the accuracy was so low.

In [24]:
# 5. TEXT SIMILARITY: Use Euclidean distance to see how similar two resumes are
from sklearn.metrics.pairwise import euclidean_distances


# Compare the very first resume (Index 0) to the second resume (Index 1) in our sample
distance = euclidean_distances(tfidf_matrix[0], tfidf_matrix[1])

print("\n=== Text Similarity (Euclidean Distance) ===")
print(f"Distance between Resume 0 and Resume 1: {distance[0][0]:.4f}")
print(f"Resume 0 Actual Category: {df_sample.loc[0, 'category']}")
print(f"Resume 1 Actual Category: {df_sample.loc[1, 'category']}")


=== Text Similarity (Euclidean Distance) ===
Distance between Resume 0 and Resume 1: 1.3752
Resume 0 Actual Category: TEACHER
Resume 1 Actual Category: APPAREL


The euclidean distance measures how similar the two resumes are. A distance of 1.3 is very high, meaning that they are very different. It is true a Teacher is very different than Apparel.